# Hypertower Repro Pipeline

This notebook documents the full run sequence used to reproduce current results.

## 0) Environment + Paths

- Activate `fundus_imaging` environment
- Run from repo root
- Confirm data paths:
  - `Papila/FundusImages`
  - `Papila/ClinicalData`
  - `Papila/ExpertsSegmentations`

In [3]:
import os
from pathlib import Path

# Walk up from cwd until we find the repo root (identified by presence of classes/v2/)
def find_repo_root(marker="classes/v2"):
    p = Path.cwd()
    for candidate in [p, *p.parents]:
        if (candidate / marker).exists():
            return candidate
    raise RuntimeError(f"Could not find repo root (looked for '{marker}' starting from {p})")

os.chdir(find_repo_root())
print("Working directory:", Path.cwd())

Working directory: /home/rpotter/hypertower


In [4]:
from pathlib import Path

required = [
    Path("Papila/FundusImages"),
    Path("Papila/ClinicalData"),
    Path("Papila/ExpertsSegmentations"),
    Path("REFUGE"),
]
for p in required:
    print(f"{p}:", "OK" if p.exists() else "MISSING")

Papila/FundusImages: OK
Papila/ClinicalData: OK
Papila/ExpertsSegmentations: OK
REFUGE: OK


## 1) Build UNet Manifest

In [5]:
!python3 scripts/main/refuge/build_manifest.py --output manifest.csv

Manifest saved to manifest.csv with 2088 entries


## 2) Train UNet Segmenter (per-image normalization)

Current tuned baseline:
- `--device cuda`
- `--batch-size 8`
- `--loader-workers 14`
- `--in-memory-cache`
- `--cache-workers 4`

In [7]:
!python3 scripts/main/refuge/run_unet_segmenter.py \
  --manifest manifest.csv \
  --train --evaluate \
  --normalize per_image \
  --train-datasets refuge --val-datasets refuge --holdout-datasets refuge \
  --epochs 40 --batch-size 8 \
  --device cuda --loader-workers 14 \
  --in-memory-cache --cache-workers 4 \
  --checkpoint-dir models/v2/refuge/segmentation/per_image \
  --eval-output analysis_data/segmenter_eval/v2_refuge_per_image \
  --eval-metrics-path analysis_data/segmenter_eval/v2_refuge_per_image/metrics.csv

[UNet] device=cuda (cuda_available=True, workers=14)
[UNet] gpu=AMD Radeon RX 7800 XT
[UNet] in_memory_cache=enabled (note: memory use scales with loader workers)
[UNetSegmenter] prebuilding in-memory cache for 1600 samples (cache_workers=4)
Warm cache:   2%|▌                        | 39/1600 [00:01<00:51, 30.28sample/s]
^C
Traceback (most recent call last):
  File "/home/rpotter/hypertower/classes/unet_segmenter.py", line 219, in prebuild_in_memory_cache
    for fut in tqdm(as_completed(futures), total=len(futures), desc="Warm cache", unit="sample"):
  File "/home/rpotter/miniconda3/envs/fundus_imaging/lib/python3.12/site-packages/tqdm/std.py", line 1181, in __iter__
    for obj in iterable:
  File "/home/rpotter/miniconda3/envs/fundus_imaging/lib/python3.12/concurrent/futures/_base.py", line 243, in as_completed
    waiter.event.wait(wait_timeout)
  File "/home/rpotter/miniconda3/envs/fundus_imaging/lib/python3.12/threading.py", line 655, in wait
    signaled = self._cond.wait(timeou

## 3) Image-Only Baseline

Single-eye CNN with no metadata input (`--bridge-mode image_only`) across three crop strategies.  
Isolates the image tower's standalone contribution.  
Outputs under `analysis_data/pipeline_imgonly_{nocrop,gt,unet}/`.

In [8]:
import subprocess
from pathlib import Path

COMMON = [
    "--tower-mode", "single",
    "--epochs", "40",
    "--n-splits", "5",
    "--backbone", "refugelike",
    "--bridge-mode", "image_only",
    "--single-warmup-tower-epochs", "4",
    "--single-warmup-fused-epochs", "0",
    "--img-crop-manifest", "manifest.csv",
]
RUNS = [
    ("pipeline_imgonly_nocrop", "binary",     []),
    ("pipeline_imgonly_nocrop", "multiclass", []),
    ("pipeline_imgonly_gt",     "binary",     ["--img-crop-gt"]),
    ("pipeline_imgonly_gt",     "multiclass", ["--img-crop-gt"]),
    ("pipeline_imgonly_unet",   "binary",     ["--img-crop-weights",
                                               "models/v2/refuge/segmentation/per_image/best.pt"]),
    ("pipeline_imgonly_unet",   "multiclass", ["--img-crop-weights",
                                               "models/v2/refuge/segmentation/per_image/best.pt"]),
]

for run_name, eval_mode, extra in RUNS:
    tm_dir = Path("analysis_data") / run_name / eval_mode / "single"
    if (tm_dir / "summary.json").exists():
        print(f"[imgonly] {run_name}/{eval_mode} already complete — skipping.")
        continue
    print(f"[imgonly] Running {run_name}/{eval_mode} ...")
    subprocess.run([
        "python", "scripts/main/v2/run_multifold_v2.py",
        "--run-name", run_name,
        "--eval-mode", eval_mode,
        *COMMON, *extra,
    ], check=True)

[imgonly] pipeline_imgonly_nocrop/binary already complete — skipping.
[imgonly] pipeline_imgonly_nocrop/multiclass already complete — skipping.
[imgonly] pipeline_imgonly_gt/binary already complete — skipping.
[imgonly] pipeline_imgonly_gt/multiclass already complete — skipping.
[imgonly] pipeline_imgonly_unet/binary already complete — skipping.
[imgonly] pipeline_imgonly_unet/multiclass already complete — skipping.


## 4) MD-Only Baseline

Metadata-only MLP for 50, 200, and 500 epochs (binary + multiclass).  
Establishes the ceiling of what clinical features alone can achieve.  
Outputs under `analysis_data/pipeline_mdonly_{50,200,500}ep/`.

In [9]:
import subprocess
from pathlib import Path

RUNS = [
    ("pipeline_mdonly_50ep",  50),
    ("pipeline_mdonly_200ep", 200),
    ("pipeline_mdonly_500ep", 500),
]
EVAL_MODES = ["binary", "multiclass"]

for run_name, epochs in RUNS:
    for eval_mode in EVAL_MODES:
        tm_dir = Path("analysis_data") / run_name / eval_mode / "single"
        if (tm_dir / "summary.json").exists():
            print(f"[mdonly] {run_name}/{eval_mode} already complete — skipping.")
            continue
        print(f"[mdonly] Running {run_name}/{eval_mode} ({epochs} epochs) ...")
        subprocess.run([
            "python", "scripts/main/v2/run_md_mlp.py",
            "--run-name", run_name,
            "--eval-mode", eval_mode,
            "--tower-mode", "single",
            "--epochs", str(epochs),
            "--n-splits", "5",
        ], check=True)

[mdonly] Running pipeline_mdonly_50ep/binary (50 epochs) ...
Device: cuda
Loading PAPILA data...
Loaded: 488 rows  feature_dim=25
[binary] rows=420

[fold 1/5] eye_train_n=320 bilat_val_n=40 holdout_n=10 warmup=2+2 total=54
  ep   1/54 [tower_warmup:0/50] loss=0.5988 acc=0.7656 val_auc=0.2370 val_acc=0.8250 best_auc=-1.0000
  ep  10/54 [main:6/50] loss=0.4603 acc=0.8094 val_auc=0.6775 val_acc=0.8250 best_auc=0.6775
  ep  20/54 [main:16/50] loss=0.4198 acc=0.8219 val_auc=0.7662 val_acc=0.8250 best_auc=0.7662
  ep  30/54 [main:26/50] loss=0.3825 acc=0.8469 val_auc=0.7587 val_acc=0.8500 best_auc=0.7695
  ep  40/54 [main:36/50] loss=0.3610 acc=0.8469 val_auc=0.7413 val_acc=0.8500 best_auc=0.7695
  ep  50/54 [main:46/50] loss=0.3039 acc=0.8812 val_auc=0.7381 val_acc=0.8375 best_auc=0.7695
  ep  54/54 [main:50/50] loss=0.3169 acc=0.8750 val_auc=0.7435 val_acc=0.8375 best_auc=0.7695
  [fold 1] best_epoch=25 best_auc=0.7695 val_auc=0.7695 val_acc=0.8375 hld_auc=0.5800 hld_acc=0.5000

[fold 2/5

## 5) Run Pipeline Experiments

Runs `single` + `ensemble` (with fused head) across `binary` and `multiclass` for three crop strategies.  
Outputs land under `analysis_data/pipeline_{nocrop,gt,unet}/`.

Each cell can be run independently; expect ~2–4 hours per crop mode on GPU.

In [ ]:
# 3a) No crop — original full-size images
!python scripts/main/v2/multirun_hypertower.py \
  --tower-modes single ensemble \
  --eval-modes binary multiclass \
  --epochs 40 --n-splits 5 \
  --backbone refugelike \
  --img-crop-manifest analysis_data/unet_manifest.csv \
  --warmup-md-epochs 50 \
  --fused-head \
  --run-name pipeline_nocrop

In [ ]:
# 3b) GT crop — expert segmentation masks crop the optic disc region
!python scripts/main/v2/multirun_hypertower.py \
  --tower-modes single ensemble \
  --eval-modes binary multiclass \
  --epochs 40 --n-splits 5 \
  --backbone refugelike \
  --img-crop-gt \
  --warmup-md-epochs 50 \
  --fused-head \
  --run-name pipeline_gt

In [ ]:
# 3c) UNet crop — trained segmenter crops the optic disc region
!python scripts/main/v2/multirun_hypertower.py \
  --tower-modes single ensemble \
  --eval-modes binary multiclass \
  --epochs 40 --n-splits 5 \
  --backbone refugelike \
  --img-crop-manifest analysis_data/unet_manifest.csv \
  --img-crop-weights models/v2/refuge/segmentation/per_image/best.pt \
  --img-crop-normalize per_image \
  --img-crop-cache analysis_data/v2_crops_unet_refuge \
  --warmup-md-epochs 50 \
  --fused-head \
  --run-name pipeline_unet

## 6) Visualizations

ROC curves, probability strips (binary), and probability triangles (multiclass) for all runs and modes.  
Outputs written to `{run_dir}/{eval_mode}/{tower_mode}/plots/`.

In [ ]:
import subprocess
from pathlib import Path

RUN_DIRS = {
    "nocrop": Path("analysis_data/pipeline_nocrop"),
    "gt":     Path("analysis_data/pipeline_gt"),
    "unet":   Path("analysis_data/pipeline_unet"),
}
EVAL_MODES  = ["binary", "multiclass"]
TOWER_MODES = ["single", "ensemble"]

for run_name, run_dir in RUN_DIRS.items():
    for eval_mode in EVAL_MODES:
        for tower_mode in TOWER_MODES:
            mode_dir = run_dir / eval_mode / tower_mode
            if not mode_dir.exists():
                print(f"  skip (not found): {mode_dir}")
                continue
            print(f"--- {run_name} / {eval_mode} / {tower_mode} ---")

            # ROC curves
            subprocess.run([
                "python", "scripts/output_analysis/visualizations/plot_run_roc_v2.py",
                "--run-dir", str(run_dir),
                "--eval-mode", eval_mode,
                "--tower-mode", tower_mode,
            ], check=True)

            # Probability strips — binary only
            if eval_mode == "binary":
                subprocess.run([
                    "python", "scripts/output_analysis/visualizations/plot_prob_strips.py",
                    "--run-dir", str(mode_dir),
                    "--head", "fused", "--style", "strips",
                ], check=True)

            # Probability triangle-3D — multiclass only
            if eval_mode == "multiclass":
                subprocess.run([
                    "python", "scripts/output_analysis/visualizations/plot_prob_strips.py",
                    "--run-dir", str(mode_dir),
                    "--head", "fused", "--style", "triangle3d",
                ], check=True)

## 7) Explainability

MD feature importance (Phase 1), GradCAM heatmaps (Phase 2), and fusion event analysis (Phase 3).  
Phase 3 is disk-based (no re-inference); Phases 1 & 2 reload the model per fold.  
Skip phases with `--no-phase1`, `--no-phase2`, or `--no-phase3`.

In [ ]:
import subprocess
from pathlib import Path

RUN_DIRS = {
    "nocrop": Path("analysis_data/pipeline_nocrop"),
    "gt":     Path("analysis_data/pipeline_gt"),
    "unet":   Path("analysis_data/pipeline_unet"),
}
EVAL_MODES  = ["binary", "multiclass"]
TOWER_MODES = ["single", "ensemble"]

for run_name, run_dir in RUN_DIRS.items():
    for eval_mode in EVAL_MODES:
        for tower_mode in TOWER_MODES:
            run_dir_path = run_dir / eval_mode / tower_mode
            if not run_dir_path.exists():
                print(f"  skip (not found): {run_dir_path}")
                continue
            print(f"--- {run_name} / {eval_mode} / {tower_mode} ---")
            subprocess.run([
                "python", "scripts/output_analysis/explainability/explain_run.py",
                "--run-dir", str(run_dir_path),
                "--n-splits", "5",
            ], check=True)